# 03 - 肿瘤外科 LoRA 微调（Qwen3-4B）⭐ 云端

**ClawTeam v3.1 真协作 Harness 版**

- **基座**：Qwen3-4B-Instruct（云端 4090，约 12GB 显存）
- **数据**：真实 Benchmark（MedQA + CMExam）筛选 + LLM 改写扩充，~2000 条
- **时间**：2-3 小时
- **成本**：~¥10

**前置准备**：
```bash
# 1. 下载 benchmark
python eval/datasets/download_medqa.py
python eval/datasets/download_cmexam.py
python eval/datasets/download_medbullets.py

# 2. 准备外科训练数据
python eval/data_generators/prepare_surgeon_data.py
```

## Step 0: 修复 Windows 编码 + HF 镜像（云端不需要镜像，本地需要）

In [ ]:
import os, sys, pathlib, builtins

# AutoDL 云端如果在国内，建议保留镜像；海外环境可注释
os.environ.setdefault('HF_ENDPOINT', 'https://hf-mirror.com')
os.environ['WANDB_DISABLED'] = 'true'
os.environ['TRANSFORMERS_NO_ADVISORY_WARNINGS'] = '1'

# Windows GBK 兼容（云端 Linux 不需要，但保留无害）
_orig_read_text = pathlib.Path.read_text
def _utf8_read_text(self, encoding=None, errors=None, **kw):
    return _orig_read_text(self, encoding=encoding or 'utf-8', errors=errors, **kw)
pathlib.Path.read_text = _utf8_read_text

_orig_open = builtins.open
def _utf8_open(file, mode='r', buffering=-1, encoding=None, errors=None, **kw):
    if 'b' not in mode and encoding is None:
        encoding = 'utf-8'
    return _orig_open(file, mode, buffering, encoding, errors, **kw)
builtins.open = _utf8_open

print('环境配置完成')

## Step 1: 环境检查

In [ ]:
import torch, transformers, peft, trl
from pathlib import Path

print(f'PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}')
print(f'Transformers: {transformers.__version__}')
print(f'PEFT: {peft.__version__}')
print(f'TRL: {trl.__version__}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## Step 2: 配置（云端 Qwen3-4B）

In [ ]:
# ===== 关键参数 =====
# 主推 Qwen3-4B（4090 24GB 完美）
# 如果 4090 紧张可用 Qwen3-1.7B-Instruct
MODEL_NAME = 'Qwen/Qwen3-4B'
# 备选: 'Qwen/Qwen3-1.7B-Instruct' / 'Qwen/Qwen3-8B-Instruct'

NOTEBOOK_DIR = Path.cwd()
EVAL_DIR = NOTEBOOK_DIR.parent
BACKEND_DIR = EVAL_DIR.parent
DATA_PATH = EVAL_DIR / 'datasets' / 'data' / 'training' / 'surgeon_train.jsonl'
OUTPUT_DIR = EVAL_DIR / 'models' / 'surgeon_qwen3_lora'
ROLE_PROMPT_PATH = BACKEND_DIR / 'workspace' / 'roles' / 'SURGEON.md'

# 训练超参
MAX_LENGTH = 1024
BATCH_SIZE = 2     # 4090 跑 4B 可以 2，跑 8B 改 1
GRAD_ACCUM = 8     # 实际 batch = 16
LEARNING_RATE = 5e-5
NUM_EPOCHS = 5
WARMUP_RATIO = 0.1

# LoRA 参数
LORA_R = 16          # Qwen3 推荐 r=16-32（比之前的 8 大）
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
TARGET_MODULES = ['q_proj', 'v_proj', 'k_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']

USE_4BIT = False  # Qwen3-4B 不需要量化；7B/8B 可开

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'数据存在: {DATA_PATH.exists()}')
print(f'输出: {OUTPUT_DIR}')

## Step 3: 加载数据

In [ ]:
import json, random
from datasets import Dataset

random.seed(42)

records = []
with open(DATA_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line:
            r = json.loads(line)
            msgs = r.get('messages', [])
            if len(msgs) >= 3 and msgs[-1].get('role') == 'assistant':
                records.append({'messages': msgs})

random.shuffle(records)
n_val = max(40, int(len(records) * 0.05))
train_dataset = Dataset.from_list(records[n_val:])
val_dataset = Dataset.from_list(records[:n_val])
print(f'训练: {len(train_dataset)}, 验证: {len(val_dataset)}')

# 数据来源分布
from collections import Counter
sources = Counter(r.get('source', 'unknown') for r in records if 'source' in r)
print(f'数据来源分布: {sources}')

## Step 4: 加载基座模型 + LoRA

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model_kwargs = {
    'dtype': torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    'device_map': 'auto',
    'trust_remote_code': True,
}

if USE_4BIT:
    model_kwargs['quantization_config'] = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, **model_kwargs)
model.config.use_cache = False

if USE_4BIT:
    model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
    target_modules=TARGET_MODULES,
    bias='none', task_type='CAUSAL_LM',
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print(f'显存占用: {torch.cuda.memory_allocated() / 1e9:.2f} GB')

## Step 5: 训练（兼容新旧版 trl）

In [ ]:
from trl import SFTTrainer, SFTConfig
import inspect

sft_init_params = set(inspect.signature(SFTConfig.__init__).parameters.keys())
config_kwargs = dict(
    output_dir=str(OUTPUT_DIR / 'checkpoints'),
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type='cosine',
    weight_decay=0.01,
    eval_strategy='steps',
    eval_steps=50,
    save_steps=100,
    save_strategy='steps',
    save_total_limit=2,
    load_best_model_at_end=True,
    logging_steps=10,
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported() and torch.cuda.is_available(),
    gradient_checkpointing=True,
    report_to='none',
    seed=42,
)
if 'max_length' in sft_init_params:
    config_kwargs['max_length'] = MAX_LENGTH
elif 'max_seq_length' in sft_init_params:
    config_kwargs['max_seq_length'] = MAX_LENGTH
if 'packing' in sft_init_params:
    config_kwargs['packing'] = False

sft_config = SFTConfig(**config_kwargs)

trainer_init_params = set(inspect.signature(SFTTrainer.__init__).parameters.keys())
trainer_kwargs = dict(
    model=model, args=sft_config,
    train_dataset=train_dataset, eval_dataset=val_dataset,
)
if 'processing_class' in trainer_init_params:
    trainer_kwargs['processing_class'] = tokenizer
elif 'tokenizer' in trainer_init_params:
    trainer_kwargs['tokenizer'] = tokenizer

trainer = SFTTrainer(**trainer_kwargs)
print('开始训练 Qwen3-4B 外科 LoRA...')
trainer.train()

## Step 6: 保存

In [ ]:
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

with open(OUTPUT_DIR / 'training_info.json', 'w', encoding='utf-8') as f:
    json.dump({
        'base_model': MODEL_NAME,
        'role': 'surgeon',
        'lora_config': {
            'r': LORA_R, 'alpha': LORA_ALPHA,
            'target_modules': TARGET_MODULES,
        },
        'train_size': len(train_dataset),
        'val_size': len(val_dataset),
        'epochs': NUM_EPOCHS,
        'max_length': MAX_LENGTH,
    }, f, ensure_ascii=False, indent=2)

print(f'保存到: {OUTPUT_DIR.resolve()}')
for p in OUTPUT_DIR.glob('*'):
    if p.is_file():
        print(f'  {p.name}: {p.stat().st_size / 1e6:.1f} MB')

## Step 7: 推理测试

In [ ]:
model.config.use_cache = True
model.eval()

system_prompt = ROLE_PROMPT_PATH.read_text(encoding='utf-8') if ROLE_PROMPT_PATH.exists() else '你是肿瘤外科医生。'

test_qs = [
    '58 岁男性，右上肺 3cm 腺癌，T2N0M0，吸烟 30 年，FEV1 75% 预计值。可手术吗？建议什么术式？',
    '65 岁女性，胃癌 T3N2M0，幽门梗阻。外科评估：可切除性 + 术式 + 围手术期管理',
    '72 岁男性，肝细胞癌 5cm，肝硬化 Child A，AFP 升高。手术 vs 介入怎么选？',
]

for q in test_qs:
    messages = [
        {'role': 'system', 'content': system_prompt},
        {'role': 'user', 'content': q},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors='pt').to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=512,
            do_sample=True, temperature=0.7, top_p=0.9,
            pad_token_id=tokenizer.pad_token_id,
        )
    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    print(f'\n=== 问: {q} ===\n答:\n{response}\n{"-"*80}')

## Step 8: Round 2 反驳能力测试（关键 — 验证真协作）

In [ ]:
# 模拟 MDT 场景：外科看到病理 + 内科 + 放疗的意见后，应该有自己的反驳/修正
round2_test = '''
【病例】
58 岁男性，肺腺癌，T3N2M0，EGFR 阳性。

【其他专家 Round 1 意见】
病理科：分化差，建议先新辅助治疗
肿瘤内科：建议先靶向（吉非替尼）2 个月，再评估手术
放疗科：建议手术后辅助放疗，剂量 50Gy

【你的 Round 1 意见】
可切除，建议右肺上叶切除 + 系统淋巴清扫

【Round 2 任务】
现在请按 Round 2 格式回答（同意 / 反对 / 修正 / 最终意见）。
'''

messages = [
    {'role': 'system', 'content': system_prompt},
    {'role': 'user', 'content': round2_test},
]
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(text, return_tensors='pt').to(model.device)
with torch.no_grad():
    outputs = model.generate(
        **inputs, max_new_tokens=600,
        do_sample=True, temperature=0.7, top_p=0.9,
        pad_token_id=tokenizer.pad_token_id,
    )
response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
print(f'\n=== Round 2 反驳测试 ===\n{response}')